In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split, ParameterGrid, cross_validate
import nltk
from nltk.tokenize import word_tokenize
from sklearn.utils import shuffle
from sklearn import decomposition
import pandas as pd
from copy import deepcopy
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import json
import itertools
import random
import warnings
import string
from shutil import copyfile
warnings.filterwarnings("ignore", category=UserWarning)

from functions import *


In [ ]:
global_shuffle_seed = 4
global_debug=True
global_override=True

In [ ]:
result, clf_result = {}, {}
df_data = json.load(open('data/data.json'))
df_data = df_data[df_data['is_flood'].notna()]
data_true = query_dataframe(df_data, {'is_flood':True})
data_false = query_dataframe(df_data, {'is_flood':False})
print('Total:',len(df_data),'True:',len(data_true), 'False:',len(data_false))

### Preprocess Data

In [31]:
custom_stop_words = set(['date', 'published'])
stop_words = set(nltk.corpus.stopwords.words('english'))
punctuations = set(string.punctuation)
all_stop_words = stop_words.union(punctuations, custom_stop_words)
def preprocess(x):
    x = re.sub('[^a-z\s]', ' ', x.lower())
    x = [w for w in x.split() if w not in all_stop_words and len(w)>3]
    return ' '.join(x)

In [32]:
df_data['org_text'] = df_data['text']
df_data['text'] = df_data['text'].apply(preprocess)

### Split Data

In [33]:
def make_data_ratio(df_data, test_size=None, train_size=None, shuffle_seed=4, debug=False, 
                    save_folder=None, load_folder=None, override=False, file_prefix=''):
    save_file, load_file=None, None
    if save_folder: save_file = os.path.join(save_folder,file_prefix+'data.json')
    if load_folder: load_file = os.path.join(load_folder,file_prefix+'data.json')
    
    if not override and load_file and os.path.isfile(load_file):
        if debug: print('loaded',load_file)
        js = json.load(open(load_file))
        train_df = pd.DataFrame(js['train'])
        test_df = pd.DataFrame(js['test'])
        return {'train':train_df, 'test':test_df}
    
    train_df, test_df = train_test_split(df_data, test_size=test_size, train_size=train_size, random_state=shuffle_seed, stratify=df_data['is_flood'])
    
    if debug: print('Data Loaded')

    if save_file:
        train_json = train_df.to_json(orient='records')
        test_json = test_df.to_json(orient='records')
        json.dump({'train':json.loads(train_json), 'test':json.loads(test_json)}, open(save_file,'w'), indent=2)
    return {'train':train_df, 'test':test_df}


In [34]:
result, clf_result = {}, {}
test_size = 0.2
if not os.path.isdir(save_data_folder): os.mkdir(save_data_folder)
debug=global_debug or False
override=global_override or False
data_split = make_data_ratio(df_data, test_size=test_size,
                               debug=debug, shuffle_seed=global_shuffle_seed, override=override)


Data Loaded


In [35]:
print('Train:',len(data_split['train']), '\t\tTest:',len(data_split['test']))
print('Train is_flood:',len(data_split['train'].loc[data_split['train']['is_flood']==True]), \
'\tTrain not is_flood:',len(data_split['train'].loc[data_split['train']['is_flood']==False]))
print('Test is_flood:',len(data_split['test'].loc[data_split['test']['is_flood']==True]), \
'\tTest not is_flood:',len(data_split['test'].loc[data_split['test']['is_flood']==False]))

Train: 1104 		Test: 276
Train is_flood: 530 	Train not is_flood: 574
Test is_flood: 133 	Test not is_flood: 143


### Classifier

In [40]:
def make_data(vect_fit, ratio):
    train, test = ratio.get('train',None), ratio.get('test',None)
    if train is None or test is None: raise Exception('Train or Test data not found')
    all_X = list(train['text'])
    
    vect = vect_fit.fit(all_X)
    trainX, testX = vect.transform(list(train['text'])), vect.transform(list(test['text']))
    trainY, testY = [1 if t else 0 for t in train['is_flood']], [1 if t else 0 for t in test['is_flood']]
    return trainX, testX, trainY, testY, vect


In [41]:
def run_classifier(clf, trainX, testX, trainY, testY):
    clf_fit = clf.fit(trainX, trainY)
    clf_pred = clf_fit.predict(testX)
    clf_acc = accuracy_score(testY, clf_pred)
    return clf_fit, clf_pred, clf_acc


In [42]:
def get_method(main_d, name):
    if name not in main_d: raise Exception('Cannot find classifier/feature_extractor name in parameter dictionary')
    d = main_d[name]
    method = d.get('method',None)
    base_method = d.get('base_method',None)
    if method and base_method: raise Exception('Cannot have method and base method both.')
    if not method and not base_method: raise Exception('Unable to parse the method from classifier/feature_extractor')
    params = d.get('params',None)
    if method:
        if params: return method, params
        else: return method, None
    if base_method:
        prev_method, prev_params = get_method(main_d, base_method)
        if params:
            for k,v in params.items(): prev_params[k] = v
        return prev_method, prev_params

def make_method(main_d, name, override_params={}):
    method, params = get_method(main_d, name)[:]
    if override_params:
        for k,v in override_params.items(): params[k] = v
    if params: return method(**params)
    else: return method()


In [43]:
def run_grid(grid, data, feature_extract, classifiers, clf_result, result, 
             debug=False, override=False, save_folder=None, load_folder=None, file_prefix=''):
    save_clf_result = {}
    vectCache, classifierCache = {}, {}
    if load_folder:
        res_file = os.path.join(load_folder,file_prefix+'clf_result.json')
        clf_res_file = os.path.join(load_folder,file_prefix+'result.json')
        if os.path.isfile(res_file): clf_result=json.load(open(res_file))
        if os.path.isfile(clf_res_file): result=json.load(open(clf_res_file))
        if os.path.isfile(res_file) and os.path.isfile(clf_res_file) and debug: print('loaded result')
    
    if override:
        clf_result, result = {}, {}
        if debug: print('OVERRIDE')
    for g in list(grid):
        try:
            feature_name = g.get('feature_extract',None)
            clf_name = g.get('classifier', None)
            if not feature_name or not clf_name:
                raise Exception('Feature Extract and Classifier Name required')
            result_key = feature_name + '-' + clf_name
            if result.get(result_key): continue
            if debug: print('Feature:', feature_name, '  Clasifier:',clf_name, '  Key:',result_key)
            
            if feature_name in vectCache:
                (trainX, testX, trainY, testY, feature2) = vectCache[feature_name]
            else:
                feature = make_method(feature_extract, feature_name)
                trainX, testX, trainY, testY, feature2 = make_data(feature, data)
                vectCache[feature_name] = (trainX, testX, trainY, testY, feature2)

            clf = make_method(classifiers, clf_name)
            clf_fit, clf_pred, clf_acc = run_classifier(clf, trainX, testX, trainY, testY)
            
            result[result_key] = {
                'feature_extract': feature_name,
                'classifier': clf_name,
                'accuracy': clf_acc
            }
            
            clf_result[result_key] = {
                'feature_extract': feature_name,
                'classifier': clf_name,
                'clf': clf_fit,
                'feature': feature2,
                'predict': clf_pred
            }
            
            save_clf_result[result_key] = {
                'feature_extract': feature_name,
                'classifier': clf_name,
                'predict': clf_pred.tolist()
            }  
        except Exception as e:
            print('Error:',e)
            continue
    if save_folder:
        json.dump(save_clf_result, open(os.path.join(load_folder,file_prefix+'clf_result.json'),'w'), indent=2)
        json.dump(result, open(os.path.join(load_folder,file_prefix+'result.json'),'w'), indent=2)
    return clf_result, result


In [44]:
def run_grid_cross_validate(grid, data, feature_extract, classifiers, result, debug=False):
    for g in list(grid):
        feature_name = g.get('feature_extract',None)
        clf_name = g.get('classifier', None)
        if not feature_name or not clf_name:
            raise Exception('Feature Extract and Classifier Name required')
        result_key = feature_name + '-' + clf_name
        if result.get(result_key): continue
        if debug: print('Feature:', feature_name, '  Clasifier:',clf_name, '  Key:',result_key)

        feature = make_method(feature_extract, 'TFIDF')
        all_X = data['text']
        all_Y = data['is_flood']
        vect = feature.fit(all_X)
        x, y = vect.transform(list(all_X)), [1 if t else 0 for t in all_Y]
        clf = make_method(classifiers, clf_name)
        cv = cross_validate(clf, x, y, cv=5,
                      scoring=('accuracy', 'precision', 'recall', 'f1'))
        result[result_key] = cv
    return result

In [45]:
def parse_result(result, clf_result=None, data=None, accuracy_threshold=None, split_by='classifier'):
    keys = list(result.keys())
    temp_df = pd.DataFrame(list(result.values()))
    if clf_result is not None and data is not None:
        presicion, recall, f1, support = [], [], [], []
        actual = [1 if f else 0 for f in list(data['test']['is_flood'])]
        for method_name in keys:
            predict = clf_result[method_name]['predict']
            pre, rec, fsc, sup = precision_recall_fscore_support(actual, predict, average='binary')
            presicion.append(pre)
            recall.append(rec)
            f1.append(fsc)
            support.append(sup)
        temp_df['f1'] = f1
        temp_df['presicion'] = presicion
        temp_df['recall'] = recall
#     temp_df['keys'] = keys
    splt_val = list(set(list(temp_df[split_by])))
    for d in splt_val:
        if accuracy_threshold:
            new_df = temp_df.loc[temp_df[split_by]==d]
            new_df = new_df.loc[new_df['accuracy']>accuracy_threshold] \
                            .drop(split_by, axis=1) \
                            .sort_values(by='accuracy',ascending=False) \
                            .reset_index(drop=True)
        else:
            new_df = temp_df.loc[temp_df[split_by]==d] \
                            .drop(split_by, axis=1) \
                            .sort_values(by='accuracy',ascending=False) \
                            .reset_index(drop=True)
        print('{}: {}'.format(split_by,d))
        print(new_df.to_markdown())
        print()

In [46]:
def compare_result(clf_result, data, method_name, conf_matrix=True, class_report=True):
    if method_name not in clf_result: raise Exception('Cannot find method')
    res = clf_result[method_name]
    new_df = data['test']
    actual = [1 if f else 0 for f in list(new_df['is_flood'])]
    new_df['predict'] = res['predict']
    if conf_matrix:
        mat = confusion_matrix(actual, res['predict'])
        plot_confusion_matrix(mat, ['Negative', 'Positive'])
        print(mat)
    if class_report: print(classification_report(actual, res['predict']))
    cr = classification_report(actual, res['predict'])
    return new_df
    

In [47]:
def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    plt.rcParams.update({'font.size': 18})
    accuracy = np.trace(cm) / np.sum(cm).astype('float')
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

In [48]:
def make_vocab(ratio):
    train, test = ratio.get('train',None), ratio.get('test',None)
    if train is None or test is None: raise Exception('Train or Test data not found')
    all_X = list(train['text']) + list(test['text'])
    
    params= {
            'tokenizer': word_tokenize,
            'stop_words': 'english',
        }
    vect = CountVectorizer(**params)
    vect = vect.fit(all_X)
    return list(vect.vocabulary_.keys())

In [127]:
# Logistic Regression
vocab = make_vocab(data_split)
feature_extract = {
    'CountVect': {
        'classifier_type': 'Count Vectorizer',
        'method': CountVectorizer,
        'params': {
            'tokenizer': word_tokenize,
            'stop_words': 'english',
            'vocabulary': vocab
        }
    },
    'CountVect-2gram':{
        'base_method': 'CountVect',
        'params':{
            'ngram_range':(1,2)
        }
    },
    'CountVect-min_df-max_df':{
        'base_method': 'CountVect',
        'params':{
            'min_df': 0.05,
            'max_df': 0.95
        }
    },
    'CountVect-2gram-min_df-max_df':{
        'base_method': 'CountVect',
        'params':{
            'min_df': 0.05,
            'max_df': 0.95,
            'ngram_range':(1,2)
        }
    },
    'TFIDF': {
        'classifier_type': 'TFIDF',
        'method': TfidfVectorizer,
        'params': {
            'tokenizer': word_tokenize,
            'stop_words': 'english',
            'vocabulary': vocab
        }
    },
    'TFIDF-2gram':{
        'base_method': 'TFIDF',
        'params':{
            'ngram_range':(1,2)
        }
    },
    'TFIDF-min_df-max_df':{
        'base_method': 'TFIDF',
        'params':{
            'min_df': 0.05,
            'max_df': 0.95
        }
    },
    'TFIDF-2gram-min_df-max_df':{
        'base_method': 'TFIDF',
        'params':{
            'min_df': 0.05,
            'max_df': 0.95,
            'ngram_range':(1,2)
        }
    }
}

classifiers = {
    'RandomForest': {
        'classifier_type':'Random Forest ',
        'method': RandomForestClassifier,
        'params':{
            'class_weight':'balanced'
        }
    },
    'LinearSVC': {
        'classifier_type': 'Linear SVC',
        'method': LinearSVC,
        'params':{
            'class_weight':'balanced'
        }
    },
    'LogRegL1':{
        'classifier_type': 'Logistic Regression L1',
        'method': LogisticRegression,
        'params':{
            'penalty': 'l1',
            'class_weight':'balanced',
            'solver': 'liblinear',
            'max_iter': 1000
        }
    },
    'LogRegL2':{
        'classifier_type': 'Logistic Regression L2',
        'method': LogisticRegression,
        'params':{
            'penalty': 'l2',
            'class_weight':'balanced',
            'solver': 'liblinear',
            'max_iter': 1000
        }
    }
}

grid_parameters = {
    'feature_extract': list(feature_extract.keys()),
    'classifier': list(classifiers.keys()),
}

grid = ParameterGrid(grid_parameters)

In [ ]:
override=global_override or False
debug=global_debug or False
save_results_folder = 'results/'
load_results_folder = 'results/'
if not os.path.isdir(save_results_folder): os.mkdir(save_results_folder)
clf_result, result = run_grid(grid, data_split, feature_extract, classifiers, clf_result, result, 
                              debug=debug, override=override, save_folder=save_results_folder, 
                             load_folder=load_results_folder)


In [ ]:
parse_result(result, clf_result=clf_result, data=data_split)

## Different test data graphs

In [49]:
vocab = make_vocab(data_split)
feature_extract = {
    'TFIDF': {
        'classifier_type': 'TFIDF',
        'method': TfidfVectorizer,
        'params': {
            'tokenizer': word_tokenize,
            'stop_words': 'english',
            'vocabulary': vocab,
            'ngram_range':(1,2),
            'min_df': 0.05,
            'max_df': 0.95
        }
    },
}

classifiers = {
    'LinearSVC': {
        'classifier_type': 'Linear SVC',
        'method': LinearSVC,
        'params':{
            'class_weight':'balanced'
        }
    },
    'LogRegL1':{
        'classifier_type': 'Logistic Regression L1',
        'method': LogisticRegression,
        'params':{
            'penalty': 'l1',
            'class_weight':'balanced',
            'solver': 'liblinear',
            'max_iter': 1000
        }
    },
    'LogRegL2':{
        'classifier_type': 'Logistic Regression L2',
        'method': LogisticRegression,
        'params':{
            'penalty': 'l2',
            'class_weight':'balanced',
            'solver': 'liblinear',
            'max_iter': 1000
        }
    },
    'RandomForest': {
        'classifier_type':'Random Forest ',
        'method': RandomForestClassifier,
        'params':{
            'class_weight':'balanced'
        }
    },
}

grid_parameters = {
    'feature_extract': list(feature_extract.keys()),
    'classifier': list(classifiers.keys()),
}

grid = ParameterGrid(grid_parameters)

In [50]:
# feature = make_method(feature_extract, 'TFIDF')
# all_X = df_data['text']
# all_Y = df_data['is_flood']
# vect = feature.fit(all_X)
# x, y = vect.transform(list(all_X)), [1 if t else 0 for t in all_Y]
# clf = make_method(classifiers, 'LinearSVC')
# cross_validate(clf, x, y, cv=5,
#               scoring=('accuracy', 'precision', 'recall', 'f1'))

In [131]:
overall_result = []
for train_size in [10,20,50,100,200,500,1000]:
    test_size = 270
    result, clf_result = {}, {}
    debug=True
    override=True
    data_split = make_data_ratio(df_data, test_size=test_size, train_size=train_size,
                               debug=debug, shuffle_seed=global_shuffle_seed, override=override)
    print(len(data_split['train']), len(data_split['test']))
    actual = [i if i==True else 0 for i in data_split['test']['is_flood']]
    clf_result, result = run_grid(grid, data_split, feature_extract, classifiers, clf_result, result, 
                              debug=debug, override=override)
    for key, val in clf_result.items():
#         if key not in overall_result: overall_result[key] = []
        predict = val['predict']
        clf_acc = accuracy_score(actual, predict)
        pre, rec, fsc, sup = precision_recall_fscore_support(actual, predict, average='binary')
        d = { 'key':key, 'train_size':train_size, 'test_size':test_size, 'accuracy':clf_acc, 'precision':pre, 
            'recall':rec, 'f1':fsc,'predict':predict, 'actual': actual
        }
        overall_result.append(d)

Data Loaded
10 270
OVERRIDE
Feature: TFIDF   Clasifier: LinearSVC   Key: TFIDF-LinearSVC
Feature: TFIDF   Clasifier: LogRegL1   Key: TFIDF-LogRegL1
Feature: TFIDF   Clasifier: LogRegL2   Key: TFIDF-LogRegL2
Data Loaded
20 270
OVERRIDE
Feature: TFIDF   Clasifier: LinearSVC   Key: TFIDF-LinearSVC
Feature: TFIDF   Clasifier: LogRegL1   Key: TFIDF-LogRegL1
Feature: TFIDF   Clasifier: LogRegL2   Key: TFIDF-LogRegL2
Data Loaded
50 270
OVERRIDE
Feature: TFIDF   Clasifier: LinearSVC   Key: TFIDF-LinearSVC
Feature: TFIDF   Clasifier: LogRegL1   Key: TFIDF-LogRegL1
Feature: TFIDF   Clasifier: LogRegL2   Key: TFIDF-LogRegL2
Data Loaded
100 270
OVERRIDE
Feature: TFIDF   Clasifier: LinearSVC   Key: TFIDF-LinearSVC
Feature: TFIDF   Clasifier: LogRegL1   Key: TFIDF-LogRegL1
Feature: TFIDF   Clasifier: LogRegL2   Key: TFIDF-LogRegL2
Data Loaded
200 270
OVERRIDE
Feature: TFIDF   Clasifier: LinearSVC   Key: TFIDF-LinearSVC
Feature: TFIDF   Clasifier: LogRegL1   Key: TFIDF-LogRegL1
Feature: TFIDF   Clasi

In [148]:
pd.DataFrame.from_dict(overall_result) \
                .drop(['predict','actual'], axis=1) \
                .sort_values(['train_size', 'accuracy'])

,key,train_size,test_size,accuracy,precision,recall,f1
1,TFIDF-LogRegL1,10,270,0.518519,0.000000,0.000000,0.000000
0,TFIDF-LinearSVC,10,270,0.814815,0.844828,0.753846,0.796748
2,TFIDF-LogRegL2,10,270,0.829630,0.833333,0.807692,0.820313
4,TFIDF-LogRegL1,20,270,0.518519,0.000000,0.000000,0.000000
3,TFIDF-LinearSVC,20,270,0.781481,0.831776,0.684615,0.751055
5,TFIDF-LogRegL2,20,270,0.796296,0.826087,0.730769,0.775510
7,TFIDF-LogRegL1,50,270,0.662963,0.634483,0.707692,0.669091
8,TFIDF-LogRegL2,50,270,0.862963,0.854962,0.861538,0.858238
6,TFIDF-LinearSVC,50,270,0.866667,0.873016,0.846154,0.859375
10,TFIDF-LogRegL1,100,270,0.740741,0.826087,0.584615,0.684685


### BERT Classifier

In [149]:
# Calculated from BERT Classifier.ipynb
overall_result_bert = [
    {'key': 'BERT-512', 'train_size': 10, 'test_size': 270, 'accuracy': 0.6148148148148148, 
     'precision': 0.6382978723404256, 'recall': 0.46153846153846156, 'f1': 0.5357142857142858},
    {'key': 'BERT-512', 'train_size': 20, 'test_size': 270, 'accuracy': 0.8333333333333334, 
     'precision': 0.8455284552845529, 'recall': 0.8, 'f1': 0.8221343873517788},
    {'key': 'BERT-512', 'train_size': 50, 'test_size': 270, 'accuracy': 0.9, 
     'precision': 0.8705035971223022, 'recall': 0.9307692307692308, 'f1': 0.899628252788104},
    {'key': 'BERT-512', 'train_size': 100, 'test_size': 270, 'accuracy': 0.9148148148148149, 
     'precision': 0.8741258741258742, 'recall': 0.9615384615384616, 'f1': 0.9157509157509157},
    {'key': 'BERT-512', 'train_size': 200, 'test_size': 270, 'accuracy': 0.9074074074074074, 
     'precision': 0.8888888888888888, 'recall': 0.9230769230769231, 'f1': 0.9056603773584906},
    {'key': 'BERT-512', 'train_size': 500, 'test_size': 270, 'accuracy': 0.9222222222222223, 
     'precision': 0.9097744360902256, 'recall': 0.9307692307692308, 'f1': 0.920152091254753},
    {'key': 'BERT-512', 'train_size': 1000, 'test_size': 270, 'accuracy': 0.9259259259259259, 
     'precision': 0.9044117647058824, 'recall': 0.9461538461538461, 'f1': 0.9248120300751879}
]
r2 = []
for i in overall_result_bert:
    i['predict'] = []
    i['actual'] = []
    r2.append(i)
overall_result_bert = r2
pd.DataFrame.from_dict(overall_result_bert) \
                .drop(['predict','actual'], axis=1) \
                .sort_values(['train_size', 'accuracy'])

,key,train_size,test_size,accuracy,precision,recall,f1
0,BERT-512,10,270,0.614815,0.638298,0.461538,0.535714
1,BERT-512,20,270,0.833333,0.845528,0.800000,0.822134
2,BERT-512,50,270,0.900000,0.870504,0.930769,0.899628
3,BERT-512,100,270,0.914815,0.874126,0.961538,0.915751
4,BERT-512,200,270,0.907407,0.888889,0.923077,0.905660
5,BERT-512,500,270,0.922222,0.909774,0.930769,0.920152
6,BERT-512,1000,270,0.925926,0.904412,0.946154,0.924812


### Weak Classifier

In [219]:
# Evaluate weak classifier
words = ['flood', 'inundat', 'cyclone']
flood_data, non_flood_data = [], []
for row in df_data.iterrows():
    text = row[1]['text']
    if any(word in text for word in keys): flood_data.append(row[1])
    else: non_flood_data.append(row[1])
weak_flood = pd.DataFrame(flood_data)
weak_not_flood = pd.DataFrame(non_flood_data)

### OVERALL CLASSIFIER

In [161]:
result_df = pd.DataFrame.from_dict(overall_result + overall_result_bert + weak_res) \
                .drop(['predict','actual'], axis=1) \
                .sort_values(['train_size', 'accuracy'])
result_df['accuracy'] = result_df['accuracy'].apply(lambda x:round(x,2))
result_df['precision'] = result_df['precision'].apply(lambda x:round(x,2))
result_df['recall'] = result_df['recall'].apply(lambda x:round(x,2))
result_df['f1'] = result_df['f1'].apply(lambda x:round(x,2))
print(result_df.reset_index(drop=True).to_latex(index=False))

\begin{tabular}{lrrrrrr}
\toprule
                   key &  train\_size &  test\_size &  accuracy &  precision &  recall &    f1 \\
\midrule
               cyclone &           0 &       1380 &      0.53 &       0.56 &    0.14 &  0.22 \\
       inundat-cyclone &           0 &       1380 &      0.69 &       0.78 &    0.49 &  0.60 \\
               inundat &           0 &       1380 &      0.69 &       0.91 &    0.41 &  0.56 \\
         flood-cyclone &           0 &       1380 &      0.73 &       0.65 &    0.95 &  0.77 \\
                 flood &           0 &       1380 &      0.73 &       0.66 &    0.93 &  0.77 \\
 flood-inundat-cyclone &           0 &       1380 &      0.73 &       0.65 &    0.97 &  0.78 \\
         flood-inundat &           0 &       1380 &      0.74 &       0.66 &    0.95 &  0.78 \\
        TFIDF-LogRegL1 &          10 &        270 &      0.52 &       0.00 &    0.00 &  0.00 \\
              BERT-512 &          10 &        270 &      0.61 &       0.64 &    0.46 &  0.5

### cross validation

In [188]:
overall_result = []
clf_result = {}
debug=True
override=True
clf_result = run_grid_cross_validate(grid, df_data, feature_extract, classifiers, clf_result, debug=debug)
for key, val in clf_result.items():
    d = { 'key':key,
         'mean_accuracy': round(np.mean(val['test_accuracy']),2), 
         'mean_precision': round(np.mean(val['test_precision']),2),
         'mean_recall': round(np.mean(val['test_recall']),2), 
         'mean_f1': round(np.mean(val['test_f1']),2),
         'accuracy':val['test_accuracy'], 'precision':val['test_precision'],
         'recall':val['test_recall'], 'f1':val['test_f1']
    }
    overall_result.append(d)

Feature: TFIDF   Clasifier: LinearSVC   Key: TFIDF-LinearSVC
Feature: TFIDF   Clasifier: LogRegL1   Key: TFIDF-LogRegL1
Feature: TFIDF   Clasifier: LogRegL2   Key: TFIDF-LogRegL2
Feature: TFIDF   Clasifier: RandomForest   Key: TFIDF-RandomForest


In [193]:
overall_result.append(
    {'key': 'BERT-512', 
     'accuracy': [0.9456521739130435, 0.9166666666666666, 
                  0.9347826086956522, 0.927536231884058, 0.9202898550724637], 
     'precision': [0.9389312977099237, 0.8506493506493507, 
                   0.9060402684563759, 0.9121621621621622, 0.9090909090909091], 
     'recall': [0.9461538461538461, 1.0, 
                0.9712230215827338, 0.9507042253521126, 0.9090909090909091], 
     'f1': [0.9425287356321839, 0.9192982456140351, 
            0.9375, 0.9310344827586207, 0.9090909090909091], 
     'mean_accuracy': 0.9289855072463767, 'mean_precision': 0.9033747976137443, 
     'mean_recall': 0.9554344004359203, 'mean_f1': 0.9278904746191496}
)


In [194]:
print(pd.DataFrame.from_dict(overall_result) \
                .drop(['accuracy','precision', 'recall', 'f1'], axis=1) \
                .sort_values(['mean_accuracy']).to_latex(index=False))
                

\begin{tabular}{lrrrr}
\toprule
                key &  mean\_accuracy &  mean\_precision &  mean\_recall &  mean\_f1 \\
\midrule
     TFIDF-LogRegL1 &       0.900000 &        0.900000 &     0.890000 &  0.89000 \\
     TFIDF-LogRegL2 &       0.920000 &        0.890000 &     0.940000 &  0.91000 \\
 TFIDF-RandomForest &       0.920000 &        0.870000 &     0.970000 &  0.92000 \\
           BERT-512 &       0.928986 &        0.903375 &     0.955434 &  0.92789 \\
    TFIDF-LinearSVC &       0.930000 &        0.910000 &     0.950000 &  0.93000 \\
\bottomrule
\end{tabular}



### 500 train_size

In [51]:
overall_result = []
train_size = 500
test_size = len(df_data)-train_size
result, clf_result = {}, {}
debug=True
override=True
data_split = make_data_ratio(df_data, test_size=test_size, train_size=train_size,
                           debug=debug, shuffle_seed=global_shuffle_seed, override=override)
print(len(data_split['train']), len(data_split['test']))
actual = [i if i==True else 0 for i in data_split['test']['is_flood']]
clf_result, result = run_grid(grid, data_split, feature_extract, classifiers, clf_result, result, 
                              debug=debug, override=override)
for key, val in clf_result.items():
#         if key not in overall_result: overall_result[key] = []
    predict = val['predict']
    clf_acc = accuracy_score(actual, predict)
    pre, rec, fsc, sup = precision_recall_fscore_support(actual, predict, average='binary')
    d = { 'key':key, 'train_size':train_size, 'test_size':test_size, 'accuracy':clf_acc, 'precision':pre, 
        'recall':rec, 'f1':fsc,'predict':predict, 'actual': actual
    }
    overall_result.append(d)

Data Loaded
500 880
OVERRIDE
Feature: TFIDF   Clasifier: LinearSVC   Key: TFIDF-LinearSVC
Feature: TFIDF   Clasifier: LogRegL1   Key: TFIDF-LogRegL1
Feature: TFIDF   Clasifier: LogRegL2   Key: TFIDF-LogRegL2
Feature: TFIDF   Clasifier: RandomForest   Key: TFIDF-RandomForest


In [204]:
overall_result.append(
    {'key': 'BERT-512', 'train_size': 500, 'test_size': 880, 'accuracy': 0.9375, 
     'precision': 0.923963133640553, 'recall': 0.9479905437352246, 'f1': 0.9358226371061844}
)

In [207]:
print(pd.DataFrame.from_dict(overall_result) \
                .drop(['predict','actual'], axis=1) \
                .sort_values(['train_size', 'accuracy']).to_latex(index=False))

\begin{tabular}{lrrrrrr}
\toprule
                key &  train\_size &  test\_size &  accuracy &  precision &    recall &        f1 \\
\midrule
     TFIDF-LogRegL1 &         500 &        880 &  0.882955 &   0.900000 &  0.851064 &  0.874848 \\
     TFIDF-LogRegL2 &         500 &        880 &  0.904545 &   0.891455 &  0.912530 &  0.901869 \\
    TFIDF-LinearSVC &         500 &        880 &  0.913636 &   0.895216 &  0.929078 &  0.911833 \\
 TFIDF-RandomForest &         500 &        880 &  0.917045 &   0.875536 &  0.964539 &  0.917885 \\
           BERT-512 &         500 &        880 &  0.937500 &   0.923963 &  0.947991 &  0.935823 \\
\bottomrule
\end{tabular}



In [214]:
overall_results = [
    {'key': 'BERT-512', 'epochs':2, 'accuracy': 0.8818181818181818, 
     'precision': 0.841541755888651, 'recall': 0.9290780141843972, 'f1': 0.8831460674157304},
    {'key': 'BERT-512', 'epochs':5, 'accuracy': 0.9375, 
     'precision': 0.9200913242009132, 'recall': 0.9527186761229315, 'f1': 0.9361207897793262},
    {'key': 'BERT-512', 'epochs':10, 'accuracy': 0.9375, 
     'precision': 0.923963133640553, 'recall': 0.9479905437352246, 'f1': 0.9358226371061844},
    {'key': 'BERT-512', 'epochs':20, 'accuracy': 0.9272727272727272, 
     'precision': 0.8945054945054945, 'recall': 0.9621749408983451, 'f1': 0.9271070615034169}
]
print(pd.DataFrame.from_dict(overall_results) \
                .sort_values(['epochs','accuracy']).to_latex(index=False))

\begin{tabular}{lrrrrr}
\toprule
      key &  epochs &  accuracy &  precision &    recall &        f1 \\
\midrule
 BERT-512 &       2 &  0.881818 &   0.841542 &  0.929078 &  0.883146 \\
 BERT-512 &       5 &  0.937500 &   0.920091 &  0.952719 &  0.936121 \\
 BERT-512 &      10 &  0.937500 &   0.923963 &  0.947991 &  0.935823 \\
 BERT-512 &      20 &  0.927273 &   0.894505 &  0.962175 &  0.927107 \\
\bottomrule
\end{tabular}



# Classify new data

In [18]:
root_folder = 'paper_data'
newspapers = ['bdnews', 'dailySun', 'prothomalo', 'dailyObserver', 'newAge', 
              'dhakaTribune', 'thedailystar', 'theIndependent', 'theNewNation']
# newspapers += ['nytimes']
# newspapers_files = [os.path.join(root_folder, n, n+'2_data.json') for n in newspapers]
newspapers_files = [os.path.join(root_folder, 'all_paper_data', n+'1_data.json') for n in newspapers]

idSet = set(df_data['doc_id'])

def classify_new_data(newspapers_files):
    new_data = pd.DataFrame()
    text_set = set()
    for i,newspapers_path in enumerate(newspapers_files):
        temp_data = json.load(open(newspapers_path))
        temp_data2 = []
        for t in temp_data:
            if t['id'] in idSet: continue
            temp_dict = t['meta']
            for k,v in t['article'].items(): temp_dict[k]=v
            temp_dict['id'] = t['id']
            temp_dict['connect_filename'] = t.get('connect_filename',None)
            temp_dict['newspaper'] = newspapers[i]
            if t['article']['text'] not in text_set: text_set.add(t['article']['text'])
            else: continue
            temp_data2.append(temp_dict)
        temp_df = pd.DataFrame(temp_data2)
        new_data = pd.concat([new_data, temp_df])
    new_data = new_data.fillna("")
    new_data = new_data[new_data['connect_filename']==""]
    print(len(new_data))
    return new_data
### New data from newspapers


### NYTIMES new data
# newspapers_files = ['../article_scraping/paper_data/thedailystar/theDailyStar3_data.json']
new_data = classify_new_data(newspapers_files)

36123


In [25]:
def loop_data_train_test(classifier, feature, data_df=None, predictions_folder = 'predictions',
                         prev_true_data_df=None, prev_false_data_df=None, save=True):
#     if data_df is None and prev_false_data_df is None: raise Exception('No df_data or prev_dalse_data_df')
#     if prev_false_data_df is not None: data_df = prev_false_data_df
    to_keep_cols = ['datePublished', 'text', 'doc_id', 'connect_filename', 'newspaper', 'is_flood']
    data_df['new_text'] = data_df['text'].apply(preprocess)
    
    test_features = feature.transform(list(data_df['new_text']))
    test_pred = classifier.predict(test_features)
    
    data_df['is_flood'] = [bool(i) for i in test_pred]
    data_df['doc_id'] = data_df['id']
    data_df = data_df[to_keep_cols]
    true_new_data = data_df.loc[data_df['is_flood']]
    false_new_data = data_df.loc[~data_df['is_flood']]
    print('Total New Data: {}\tTrue new Data: {}'.format(len(data_df), len(true_new_data)))
    
    if prev_true_data_df is not None: df_true_new_data = pd.concat([prev_true_data_df, true_new_data])
    else: df_true_new_data = true_new_data
    js = df_true_new_data.to_json(orient='records')
    if save: json.dump(json.loads(js), open(os.path.join(predictions_folder, 'predicted_isflood.json'), 'w'), indent=2)
    
    if prev_false_data_df is not None: df_false_new_data = pd.concat([prev_false_data_df, false_new_data])
    else: df_false_new_data = false_new_data
    js = df_false_new_data.to_json(orient='records')
    if save: json.dump(json.loads(js), open(os.path.join(predictions_folder, 'predicted_not_isflood.json'), 'w'), indent=2)
    return df_true_new_data, false_new_data

In [26]:
key = 'TFIDF-LinearSVC'
feature = clf_result[key]['feature']
classifier = clf_result[key]['clf']
prev_true_data_df, prev_false_data_df = None, None
# prev_true_data_df, prev_false_data_df = get_new_predicted_data()
save = False
df_true_new_data, false_new_data = loop_data_train_test(classifier, feature, new_data, 'predictions',
                                                        prev_true_data_df, prev_false_data_df, save=save)

Total New Data: 344709	True new Data: 1873
